# Periodic committor experiment

This is the only notebook to use for the periodic-committor experiment. Do not copy or edit its cells.

## One-time setup: save the GitHub token in Colab

In Colab's left sidebar, click the **key** icon (**Secrets**) → **Add new secret**. Set the name to `GITHUB_TOKEN`, paste the repository-read token as its value, and enable **Notebook access**. This is the only time you need to handle the token. The notebook reads it only to clone the private repository in a fresh runtime; it does not write the token to the repository or to Google Drive.

## Each experiment run

1. Choose **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the setup cell and then the CPU-preflight cell. The preflight is a complete but tiny three-method run; it uses CPU computation and writes its own output folder to Drive.
3. If the preflight completes, run the GPU-pilot cell. It is the full comparison: three methods, 6,000 epochs each, and 128/256-grid finite-difference references.
4. Run the display cell after the pilot finishes. Use the final cell only to resume an interrupted pilot.

Every output directory records its configuration, code revision, runtime metadata, checkpoints, metrics, reference data, summary, and figures.

In [ ]:
from pathlib import Path
import base64
import shutil
import subprocess

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'committor-colab'

if not (REPO / 'pyproject.toml').is_file():
    if REPO.exists():
        shutil.rmtree(REPO)  # only an incomplete clone at this fixed Colab path
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Add GITHUB_TOKEN in Colab Secrets, enable Notebook access, then rerun this cell.')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    try:
        subprocess.run(
            ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}',
             'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
    finally:
        del token, auth
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
else:
    print('Using the existing clone at', REPO)

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Setup and unit tests passed. Code revision:', revision)

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess

preflight_name = 'preflight_' + datetime.now().strftime('%Y%m%d_%H%M%S')
PREFLIGHT_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{preflight_name}'
Path(PREFLIGHT_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preflight results:', PREFLIGHT_ROOT)

subprocess.run([
    'met-pinn-periodic-comparison',
    'configs/periodic_committor/preflight_comparison_cpu.json',
    '--output-dir', PREFLIGHT_ROOT,
], check=True)
print('CPU preflight completed. The next cell is now safe to use for the GPU pilot.')

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The GPU pilot requires a T4 GPU. Select it under Runtime → Change runtime type, then rerun setup and preflight.')

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('GPU pilot results:', RUN_ROOT)

subprocess.run([
    'met-pinn-periodic-comparison',
    'configs/periodic_committor/overnight_comparison.json',
    '--output-dir', RUN_ROOT,
], check=True)

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## Resume an interrupted GPU pilot

Rerun the setup cell in a fresh runtime. Then replace the quoted folder below with the `RUN_ROOT` printed by the interrupted pilot and execute the cell. Each method resumes from its `last.pt` checkpoint.

In [ ]:
# RUN_ROOT = '/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/periodic_committor_YYYYMMDD_HHMMSS'
# subprocess.run([
#     'met-pinn-periodic-comparison',
#     'configs/periodic_committor/overnight_comparison.json',
#     '--output-dir', RUN_ROOT,
#     '--resume',
# ], check=True)